In [ ]:
import os
import random
from glob import glob

import numpy as np
import cv2
import matplotlib.pyplot as plt
import yaml

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Lambda, Concatenate, Conv2D, Multiply, Add, GlobalAveragePooling2D, Reshape

print("TensorFlow:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("GPU memory growth could not be set:", e)


def load_config(path="../configs/config.yaml"):
    with open(path, "r") as f:
        return yaml.safe_load(f)

CFG = load_config()


In [ ]:
SEED = int(CFG.get("seed", 10))
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

MODE = "train"  # "train" or "pretrained"

DATA_ROOT = input("Enter dataset root path (contains train/val/test with low/high):\n> ").strip().strip('"').strip("'")
if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(f"Dataset root does not exist: {DATA_ROOT}")

IMAGE_SIZE = int(CFG["data"]["image_size"])
BATCH_SIZE = int(CFG["data"]["batch_size"])

PRETRAINED_PATH = ""
if MODE == "pretrained":
    PRETRAINED_PATH = input("Enter pretrained model path (SavedModel folder):\n> ").strip().strip('"').strip("'")
    if not os.path.isdir(PRETRAINED_PATH):
        raise FileNotFoundError(f"Pretrained model path does not exist: {PRETRAINED_PATH}")


In [ ]:
def read_image(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image.set_shape([None, None, 3])
    return tf.cast(image, tf.float32) / 255.0

def random_crop(low_image, high_image):
    shape = tf.shape(low_image)[:2]
    w = tf.random.uniform((), maxval=shape[1] - IMAGE_SIZE + 1, dtype=tf.int32)
    h = tf.random.uniform((), maxval=shape[0] - IMAGE_SIZE + 1, dtype=tf.int32)
    return low_image[h:h+IMAGE_SIZE, w:w+IMAGE_SIZE], high_image[h:h+IMAGE_SIZE, w:w+IMAGE_SIZE]

def load_pair(low_path, high_path):
    low = read_image(low_path)
    high = read_image(high_path)
    return random_crop(low, high)

def make_dataset(low_paths, high_paths):
    ds = tf.data.Dataset.from_tensor_slices((low_paths, high_paths))
    ds = ds.map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_low = sorted(glob(os.path.join(DATA_ROOT, "train", "low", "*")))
train_high = sorted(glob(os.path.join(DATA_ROOT, "train", "high", "*")))
val_low = sorted(glob(os.path.join(DATA_ROOT, "val", "low", "*")))
val_high = sorted(glob(os.path.join(DATA_ROOT, "val", "high", "*")))
test_low = sorted(glob(os.path.join(DATA_ROOT, "test", "low", "*")))
test_high = sorted(glob(os.path.join(DATA_ROOT, "test", "high", "*")))

print("Train:", len(train_low), len(train_high))
print("Val  :", len(val_low), len(val_high))
print("Test :", len(test_low), len(test_high))

if len(train_low) == 0 or len(train_high) == 0:
    raise RuntimeError("Training set is empty. Check folder structure.")
if len(val_low) == 0 or len(val_high) == 0:
    raise RuntimeError("Validation set is empty. Check folder structure.")

train_ds = make_dataset(train_low, train_high)
val_ds = make_dataset(val_low, val_high)


In [ ]:
def selective_kernel_feature_fusion(f1, f2, f3):
    c = K.int_shape(f1)[-1]
    combined = Add()([f1, f2, f3])
    gap = GlobalAveragePooling2D()(combined)
    gap = Reshape((1, 1, c))(gap)

    compact = Conv2D(filters=c // 8, kernel_size=1, activation="relu")(gap)

    w1 = Conv2D(c, kernel_size=1, activation="softmax")(compact)
    w2 = Conv2D(c, kernel_size=1, activation="softmax")(compact)
    w3 = Conv2D(c, kernel_size=1, activation="softmax")(compact)

    out = Add()([Multiply()([f1, w1]), Multiply()([f2, w2]), Multiply()([f3, w3])])
    return out

def spatial_attention_block(x):
    avg = Lambda(lambda t: K.mean(t, axis=-1, keepdims=True))(x)
    mx = Lambda(lambda t: K.max(t, axis=-1, keepdims=True))(x)
    concat = Concatenate(axis=-1)([avg, mx])
    attn = Conv2D(1, kernel_size=1, activation="sigmoid")(concat)
    return Multiply()([x, attn])

def channel_attention_block(x):
    c = x.shape[-1]
    gap = GlobalAveragePooling2D()(x)
    gap = Reshape((1, 1, c))(gap)
    fc = Conv2D(filters=c // 8, kernel_size=1, activation="relu")(gap)
    fc = Conv2D(filters=c, kernel_size=1, activation="sigmoid")(fc)
    return Multiply()([x, fc])

def dual_attention_unit_block(x):
    c = list(x.shape)[-1]
    feat = Conv2D(c, kernel_size=3, padding="same", activation="relu")(x)
    feat = Conv2D(c, kernel_size=3, padding="same")(feat)
    ca = channel_attention_block(feat)
    sa = spatial_attention_block(feat)
    merged = Concatenate(axis=-1)([ca, sa])
    merged = Conv2D(c, kernel_size=1)(merged)
    return Add()([x, merged])

def down_sampling_module(x):
    c = list(x.shape)[-1]
    main = Conv2D(c, kernel_size=1, activation="relu")(x)
    main = Conv2D(c, kernel_size=3, padding="same", activation="relu")(main)
    main = layers.MaxPooling2D()(main)
    main = Conv2D(c * 2, kernel_size=1)(main)

    skip = layers.MaxPooling2D()(x)
    skip = Conv2D(c * 2, kernel_size=1)(skip)
    return Add()([skip, main])

def up_sampling_module(x):
    c = list(x.shape)[-1]
    main = Conv2D(c, kernel_size=1, activation="relu")(x)
    main = Conv2D(c, kernel_size=3, padding="same", activation="relu")(main)
    main = layers.UpSampling2D()(main)
    main = Conv2D(c // 2, kernel_size=1)(main)

    skip = layers.UpSampling2D()(x)
    skip = Conv2D(c // 2, kernel_size=1)(skip)
    return Add()([skip, main])

def multi_scale_residual_block(x, c):
    l1 = x
    l2 = down_sampling_module(x)
    l3 = down_sampling_module(l2)

    d1 = dual_attention_unit_block(l1)
    d2 = dual_attention_unit_block(l2)
    d3 = dual_attention_unit_block(l3)

    f1 = selective_kernel_feature_fusion(d1, up_sampling_module(d2), up_sampling_module(up_sampling_module(d3)))
    f2 = selective_kernel_feature_fusion(down_sampling_module(d1), d2, up_sampling_module(d3))
    f3 = selective_kernel_feature_fusion(down_sampling_module(down_sampling_module(d1)), down_sampling_module(d2), d3)

    u1 = dual_attention_unit_block(f1)
    u2 = up_sampling_module(dual_attention_unit_block(f2))
    u3 = up_sampling_module(up_sampling_module(dual_attention_unit_block(f3)))

    fused = selective_kernel_feature_fusion(u1, u2, u3)
    out = Conv2D(c, kernel_size=3, padding="same")(fused)
    return Add()([x, out])

def recursive_residual_group(x, num_mrb, c):
    conv = Conv2D(c, kernel_size=3, padding="same")(x)
    for _ in range(num_mrb):
        conv = multi_scale_residual_block(conv, c)
    conv = Conv2D(c, kernel_size=3, padding="same")(conv)
    return Add()([x, conv])

def mirnet_model(num_rrg, num_mrb, channels):
    inp = keras.Input(shape=[None, None, 3])
    x = Conv2D(channels, kernel_size=3, padding="same")(inp)
    for _ in range(num_rrg):
        x = recursive_residual_group(x, num_mrb, channels)
    out = Conv2D(3, kernel_size=3, padding="same")(x)
    return keras.Model(inp, Add()([inp, out]))

NUM_RRG = int(CFG["model"]["num_rrg"])
NUM_MRB = int(CFG["model"]["num_mrb"])
CHANNELS = int(CFG["model"]["channels"])

model = mirnet_model(NUM_RRG, NUM_MRB, CHANNELS)
model.summary()


In [ ]:
def charbonnier_loss(y_true, y_pred):
    return tf.reduce_mean(tf.sqrt(tf.square(y_true - y_pred) + 1e-6))

def psnr_metric(y_true, y_pred):
    return tf.image.psnr(y_pred, y_true, max_val=1.0)

def ssim_metric(y_true, y_pred):
    return tf.image.ssim(y_pred, y_true, max_val=1.0)

LR = float(CFG["train"]["lr"])
EPOCHS = int(CFG["train"]["epochs"])

if MODE == "train":
    model.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss=charbonnier_loss,
        metrics=[psnr_metric, ssim_metric],
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_psnr_metric",
                factor=float(CFG["train"]["reduce_lr_factor"]),
                patience=int(CFG["train"]["reduce_lr_patience"]),
                verbose=1,
                min_delta=float(CFG["train"]["reduce_lr_min_delta"]),
                mode="max",
            )
        ],
    )

    SAVE_DIR = os.path.join("..", "pretrained", "baseline_savedmodel")
    os.makedirs(SAVE_DIR, exist_ok=True)
    model.save(SAVE_DIR)
    print("Saved model to:", SAVE_DIR)

else:
    model = tf.keras.models.load_model(PRETRAINED_PATH, compile=False)
    print("Loaded pretrained model from:", PRETRAINED_PATH)
